# 03: Loading Data into Neo4j Graph

This notebook demonstrates how to load entities and relationships into Neo4j graph database.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup
- Optionally completed [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb) to extract data first (or extract in this notebook)

All environment detection, Neo4j connection, and configuration are handled in `00-import.ipynb`.

## Overview

This notebook takes the extracted information from your notes and stores it in Neo4j as a knowledge graph. This creates a searchable network where you can find connections between different pieces of information.

We'll:
1. Set up graph schema (indexes and constraints)
2. Extract entities and relationships from NotePlan files (or use pre-extracted data)
3. Store entities and relationships in Neo4j graph


## Setup

Import libraries and set up connections. All settings, dependencies, and paths are already configured in `00-import.ipynb`.


In [8]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
import asyncio
from knowledge_agents.agents.graph_builder_agent import run_graph_builder_agent

# NotePlan utilities
from notes.traversal import get_files_from_last_month
from notes.parser import read_noteplan_file
from notes.filter import should_skip_file

# Neo4j
from neo4j import GraphDatabase

print("✅ Additional libraries imported")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded
✅ Repository components imported
🔑 Found NEO4J_PASSWORD env var (length: 5, preview: a***n)
⚠️  Auto-correcting: NEO4J_PASSWORD is 'admin' but should be 'admin123'
   Overriding to correct password. To fix permanently: export NEO4J_PASSWORD=admin123
🔍 Runtime detection: local
🔍 Environment variables: NEO4J_URI=bolt://host.docker.internal:7687, NEO4J_PASSWORD=set, LITELLM_PROXY_HOST=not set
✅ Final Settings values: neo4j_uri=bolt://localhost:7687, litellm_proxy_host=localhost
   Neo4j credentials: username=neo4j, password=******** (length: 8)
   💡 If auth fails, override password: settings = get_settings(neo4j_password='your_actual_password')
✅ Settings loaded - 💻 Mac (Local)
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge

## Setup Graph Schema

Set up indexes and constraints for the graph database.


In [10]:
# Set up graph schema (indexes and constraints)
def setup_graph_schema(driver, database: str = "neo4j"):
    """Set up Neo4j graph schema: indexes and constraints."""
    with driver.session(database=database) as session:
        # Create constraint for unique file paths
        session.run("""
            CREATE CONSTRAINT note_file_path IF NOT EXISTS
            FOR (n:Note) REQUIRE n.file_path IS UNIQUE
        """)
        
        # Create constraint for unique entity names
        session.run("""
            CREATE CONSTRAINT entity_name IF NOT EXISTS
            FOR (e:Entity) REQUIRE e.name IS UNIQUE
        """)
        
        # Create index on entity type
        session.run("""
            CREATE INDEX entity_type IF NOT EXISTS
            FOR (e:Entity) ON (e.type)
        """)
        
        print("✅ Graph schema set up (constraints and indexes)")

setup_graph_schema(driver, settings.neo4j_database)


✅ Graph schema set up (constraints and indexes)


## Extract or Load Data

You can either:
1. Extract data now (using graph builder agent)
2. Use data extracted from notebook [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb)

We'll extract data here for this example.

**Note**: This cell uses async functions. The `nest_asyncio` package (installed in the conda environment) allows `asyncio.run()` to work in Jupyter notebooks.


### Step 1: Get File Paths

Get the list of NotePlan files to process.

In [11]:
# Get NotePlan files from the last month
files = get_files_from_last_month(NOTEPLAN_DIR)
print(f"Found {len(files)} files to process")

# Filter out files we should skip
files = [(fp, mod_time) for fp, mod_time in files if not should_skip_file(fp)]
print(f"After filtering: {len(files)} files")

# Limit to first 5 files for demo (remove this limit for full processing)
files = files[:5]
print(f"Processing {len(files)} files for this demo")


Found 176 files to process
After filtering: 176 files
Processing 5 files for this demo


In [ ]:
# Enable nested event loops for Jupyter notebooks
# This allows asyncio.run() to work even when an event loop is already running
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    print("⚠️  nest_asyncio not installed. Install with: pip install nest-asyncio")
    print("   You may encounter 'asyncio.run() cannot be called from a running event loop' errors")

# Use utility function from knowledge_agents package
from knowledge_agents.utils.graph_utils import extract_from_note_file

# Process first file as a test
if files:
    file_path, mod_time = files[0]
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    print(f"Testing extraction on: {relative_path}")
    
    path, test_output = asyncio.run(extract_from_note_file(file_path, relative_path, dependencies))
    
    if test_output:
        print(f"\n✅ Extraction successful!")
        print(f"   Entities: {len(test_output.entities)}")
        print(f"   Relationships: {len(test_output.relationships)}")
        print(f"   Insights: {len(test_output.insights)}")
        
        # Show sample entities
        if test_output.entities:
            print("\n📋 Sample Entities:")
            for entity in test_output.entities[:5]:
                print(f"   - {entity.name} ({entity.type})")
        
        # Show sample relationships
        if test_output.relationships:
            print("\n🔗 Sample Relationships:")
            for rel in test_output.relationships[:3]:
                print(f"   - {rel.from_entity} --[{rel.type}]--> {rel.to_entity}")
    else:
        print("❌ Extraction failed - check error messages above")


### Step 3: Extract from All Files

Now that we've verified the extraction works, process all files.


In [ ]:
# Process all files
# nest_asyncio is already applied above, so asyncio.run() will work
extracted_data = []
for file_path, mod_time in files:
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    print(f"Processing: {relative_path}")
    
    path, output = asyncio.run(extract_from_note(file_path, relative_path))
    if output:
        extracted_data.append((path, output))
        print(f"  ✅ Extracted {len(output.entities)} entities, {len(output.relationships)} relationships")
    else:
        print(f"  ⚠️  Skipped (extraction failed)")

print(f"\n✅ Extracted data from {len(extracted_data)} files successfully")


## Load Entities and Relationships into Neo4j

Store the extracted entities and relationships in the graph database.


In [ ]:
# Use utility function from knowledge_agents package
from knowledge_agents.utils.graph_utils import create_graph_nodes_and_relationships

# Store all extracted data
total_entities = 0
total_relationships = 0

for file_path, output in extracted_data:
    entities, relationships = create_graph_nodes_and_relationships(
        driver, file_path, output, settings.neo4j_database
    )
    total_entities += entities
    total_relationships += relationships

print(f"\n✅ Loaded into Neo4j:")
print(f"   {total_entities} entities")
print(f"   {total_relationships} relationships")


## Verify Data

Check what we've stored in Neo4j.


In [ ]:
# Count nodes and relationships
with driver.session(database=settings.neo4j_database) as session:
    result = session.run("""
        MATCH (n)
        RETURN labels(n)[0] as label, COUNT(n) as count
        ORDER BY count DESC
    """)
    
    print("Node Counts:")
    for record in result:
        print(f"  {record['label']}: {record['count']} nodes")
    
    rel_result = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) as rel_type, COUNT(r) as count
        ORDER BY count DESC
    """)
    
    print("\nRelationship Counts:")
    for record in rel_result:
        print(f"  {record['rel_type']}: {record['count']} relationships")


## Next Steps

Now that graph data is loaded, proceed to:
- [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb): Generate vector embeddings from NotePlan notes
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store vector embeddings in Neo4j
- [**05-querying-graph.ipynb**](./05-querying-graph.ipynb): Query the graph with graph patterns
- [**06-querying-vector-embeddings.ipynb**](./06-querying-vector-embeddings.ipynb): Query using vector similarity search
- [**07-knn-graphrag.ipynb**](./07-knn-graphrag.ipynb): Use KNN for graph-powered RAG
